# Task 3 Data Preparation

In [18]:
import pandas as pd
import numpy as np

In [19]:
train_df = pd.read_csv('data/training_set_VU_DM.csv')
test_df = pd.read_csv('data/test_set_VU_DM.csv')

In [20]:
train_df.head(5)

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


In [21]:
datasets = {
    "Training set": train_df,
}

for name, df in datasets.items():
    nan_per_column = df.isna().sum()
    nan_per_column = nan_per_column[nan_per_column > 0].sort_values(ascending=False)

    print(f"\n{name}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    print(f"Total NaN values: {df.isna().sum().sum():,}")
    print(f"Columns with NaN values: {len(nan_per_column)}")

    display(
        pd.DataFrame({
            "NaN count": nan_per_column,
            "NaN percentage": (nan_per_column / len(df) * 100).round(2)
        })
    )



Training set
Shape: 4,958,347 rows x 54 columns
Total NaN values: 119,733,280
Columns with NaN values: 31


,NaN count,NaN percentage
comp1_rate_percent_diff,4863908,98.10
comp6_rate_percent_diff,4862173,98.06
comp1_rate,4838417,97.58
comp1_inv,4828788,97.39
comp4_rate_percent_diff,4827261,97.36
gross_bookings_usd,4819957,97.21
comp7_rate_percent_diff,4819832,97.21
comp6_rate,4718190,95.16
visitor_hist_starrating,4706481,94.92
visitor_hist_adr_usd,4705359,94.90


In [22]:
#position - old expedia ranking, not a hotel/user feature?
LEAK_COLS  = ["click_bool", "booking_bool", "gross_bookings_usd", "position"]
y_train = train_df[["click_bool", "booking_bool"]]
train_df = train_df.drop(columns=LEAK_COLS)

## missing values

In [23]:
print("\nMissing value rates (train) – columns with >0% missing:")
miss = train_df.isnull().mean().sort_values(ascending=False)
print(miss[miss > 0].to_string())


Missing value rates (train) – columns with >0% missing:
comp1_rate_percent_diff      0.980954
comp6_rate_percent_diff      0.980604
comp1_rate                   0.975813
comp1_inv                    0.973871
comp4_rate_percent_diff      0.973563
comp7_rate_percent_diff      0.972064
comp6_rate                   0.951565
visitor_hist_starrating      0.949204
visitor_hist_adr_usd         0.948977
comp6_inv                    0.947366
comp4_rate                   0.938008
comp7_rate                   0.936401
srch_query_affinity_score    0.935986
comp4_inv                    0.930690
comp7_inv                    0.928117
comp3_rate_percent_diff      0.904646
comp2_rate_percent_diff      0.887818
comp8_rate_percent_diff      0.876021
comp5_rate_percent_diff      0.830367
comp3_rate                   0.690565
comp3_inv                    0.667028
comp8_rate                   0.613449
comp8_inv                    0.599160
comp2_rate                   0.591664
comp2_inv                    0.

### Missing flag columns - where None values has meaning

In [24]:
"""
visitor_hist_starrating - no previous star-rating history
visitor_hist_adr_usd - no previous price history
srch_query_affinity_score - No search affinity data
orig_destination_distance - No user history
prop_location_score2 - Distance could not be calculated
"""
train_df["visitor_hist_starrating_missing"] = train_df["visitor_hist_starrating"].isna().astype(int)
train_df["visitor_hist_adr_usd_missing"] = train_df["visitor_hist_adr_usd"].isna().astype(int)
train_df["srch_query_affinity_score_missing"] = train_df["srch_query_affinity_score"].isna().astype(int)
train_df["orig_destination_distance_missing"] = train_df["orig_destination_distance"].isna().astype(int)
train_df["prop_location_score2_missing"] = train_df["prop_location_score2"].isna().astype(int)

In [25]:
#impute values so the model is able to use the data

#Normal values are positive, so -1 indicates unkown 
train_df["visitor_hist_starrating"] = train_df["visitor_hist_starrating"].fillna(-1)
train_df["visitor_hist_adr_usd"] = train_df["visitor_hist_adr_usd"].fillna(-1)

#because the scores are log probabilities, impute with min()-1. Closer to 0 -> better score
affinity_fill = train_df["srch_query_affinity_score"].min() - 1
train_df["srch_query_affinity_score"] = train_df["srch_query_affinity_score"].fillna(affinity_fill)

#fill inn with median, only 30% missing data
distance_median = train_df["orig_destination_distance"].median()
train_df["orig_destination_distance"] = train_df["orig_destination_distance"].fillna(distance_median)

#fill with median, only 20% missing
location_fill = train_df["prop_location_score2"].median()
train_df["prop_location_score2"] = train_df["prop_location_score2"].fillna(location_fill)

#impute with median, only 0.01 % missing
median_review = train_df["prop_review_score"].median()
train_df["prop_review_score"] = train_df["prop_review_score"].fillna(median_review)

### competitor columns

In [ ]:
"""contain a lot of missing values, but None data is meaningful: we do not have competitor data for this hotel"""

comp_rate_cols = [f"comp{i}_rate" for i in range(1, 9)]
comp_inv_cols = [f"comp{i}_inv" for i in range(1, 9)]
comp_pct_cols = [f"comp{i}_rate_percent_diff" for i in range(1, 9)]

# 1. First create missing-count features
train_df["comp_rate_missing_count"] = train_df[comp_rate_cols].isna().sum(axis=1)
train_df["comp_inv_missing_count"] = train_df[comp_inv_cols].isna().sum(axis=1)
train_df["comp_pct_missing_count"] = train_df[comp_pct_cols].isna().sum(axis=1)

# 2. Then create individual missing flags
for i in range(1, 9):
    rate_col = f"comp{i}_rate"
    inv_col = f"comp{i}_inv"
    pct_col = f"comp{i}_rate_percent_diff"

    train_df[f"{rate_col}_missing"] = train_df[rate_col].isna().astype(int)
    train_df[f"{inv_col}_missing"] = train_df[inv_col].isna().astype(int)
    train_df[f"{pct_col}_missing"] = train_df[pct_col].isna().astype(int)

# 3. Then fill missing values
for i in range(1, 9):
    rate_col = f"comp{i}_rate"
    inv_col = f"comp{i}_inv"
    pct_col = f"comp{i}_rate_percent_diff"

    train_df[rate_col] = train_df[rate_col].fillna(-2)
    train_df[inv_col] = train_df[inv_col].fillna(-2)
    train_df[pct_col] = train_df[pct_col].fillna(0)

train_df["comp_expedia_cheaper_count"] = train_df[comp_rate_cols].eq(1).sum(axis=1)
train_df["comp_expedia_more_expensive_count"] = train_df[comp_rate_cols].eq(-1).sum(axis=1)
train_df["comp_same_price_count"] = train_df[comp_rate_cols].eq(0).sum(axis=1)
train_df["comp_no_data_count"] = train_df[comp_rate_cols].eq(-2).sum(axis=1)

train_df["comp_unavailable_count"] = train_df[comp_inv_cols].eq(1).sum(axis=1)
train_df["comp_available_count"] = train_df[comp_inv_cols].eq(0).sum(axis=1)
train_df["comp_inv_no_data_count"] = train_df[comp_inv_cols].eq(-2).sum(axis=1)

### missing values in test set

In [ ]:
#missing flag columns
test_df["visitor_hist_starrating_missing"] = test_df["visitor_hist_starrating"].isna().astype(int)
test_df["visitor_hist_adr_usd_missing"] = test_df["visitor_hist_adr_usd"].isna().astype(int)
test_df["srch_query_affinity_score_missing"] = test_df["srch_query_affinity_score"].isna().astype(int)
test_df["orig_destination_distance_missing"] = test_df["orig_destination_distance"].isna().astype(int)
test_df["prop_location_score2_missing"] = test_df["prop_location_score2"].isna().astype(int)


#same imputations, conclusions and median calculations made from training set
test_df["visitor_hist_starrating"] = test_df["visitor_hist_starrating"].fillna(-1)
test_df["visitor_hist_adr_usd"] = test_df["visitor_hist_adr_usd"].fillna(-1)
test_df["srch_query_affinity_score"] = test_df["srch_query_affinity_score"].fillna(affinity_fill)
test_df["orig_destination_distance"] = test_df["orig_destination_distance"].fillna(distance_median)
test_df["prop_location_score2"] = test_df["prop_location_score2"].fillna(location_fill)
test_df["prop_review_score"] = test_df["prop_review_score"].fillna(median_review)

#comp columns
test_df["comp_rate_missing_count"] = test_df[comp_rate_cols].isna().sum(axis=1)
test_df["comp_inv_missing_count"] = test_df[comp_inv_cols].isna().sum(axis=1)
test_df["comp_pct_missing_count"] = test_df[comp_pct_cols].isna().sum(axis=1)

for i in range(1, 9):
    rate_col = f"comp{i}_rate"
    inv_col = f"comp{i}_inv"
    pct_col = f"comp{i}_rate_percent_diff"

    test_df[f"{rate_col}_missing"] = test_df[rate_col].isna().astype(int)
    test_df[f"{inv_col}_missing"] = test_df[inv_col].isna().astype(int)
    test_df[f"{pct_col}_missing"] = test_df[pct_col].isna().astype(int)

for i in range(1, 9):
    rate_col = f"comp{i}_rate"
    inv_col = f"comp{i}_inv"
    pct_col = f"comp{i}_rate_percent_diff"

    test_df[rate_col] = test_df[rate_col].fillna(-2)
    test_df[inv_col] = test_df[inv_col].fillna(-2)
    test_df[pct_col] = test_df[pct_col].fillna(0)


## Feature engineering

In [29]:
train_df["prop_review_score_is_zero"] = (
    train_df["prop_review_score"].eq(0).astype(int)
)